# Agents — Exercise 2

In Exercise 1, every call to `llm.invoke()` was a single round-trip: one prompt in, one response out, and then the interaction ended.

An **agent** changes this pattern by placing the model inside a loop. Instead of generating a single response, the model can call tools, observe their results, and continue reasoning until the task is complete.


## Setup — imports

One new import this session: `ToolMessage`. When the agent calls a tool and gets a result back,
that result is wrapped in a `ToolMessage` and appended to the history — so the model can see what happened.

```
SystemMessage   developer → model    role, constraints, tool descriptions
HumanMessage    user → model         task or question
AIMessage       model → developer    response, may include tool_calls
ToolMessage     tool → model         result of a tool call, linked by tool_call_id
```


In [1]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from llm_config import get_llm

/home/ubuntu/tutorial_rpa/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## Background — what is an agent?

A plain LLM call is stateless: text in, text out, done.
An **agent** is a loop that lets the model decide to call tools, observe the results,
and keep reasoning until it has a final answer.

An agent has two main parts:

1. **The brain** — the LLM, which performs reasoning and decision-making.
2. **The body** — the tools and interfaces that allow the agent to interact with its environment.

The **ReAct** pattern (Reason + Act) structures each step as:

```
Thought:      I need to know the turtle's current pose.
Action:       get_pose
Action Input: {}
Observation:  x=5.5, y=5.5, heading=0°
Thought:      I now have the pose. I can plan the move.
Final Answer: ...
```

The loop continues until the model emits `Final Answer:`. Everything the model has seen —
every observation — lives in the context window. There is no hidden state. **Memory = message history.**

his cycle is also described as **Think/Reason → Action → Observation**:


## Exercise 2a — The agent loop by hand *(demo — run, don't modify)*

Before using any framework, this cell builds the ReAct loop manually: the model outputs
`Thought:` / `Action:` lines as plain text, we parse them with string matching, call the right
function, and feed the result back as an `Observation:`.

This approach works, but it is fragile — one unexpected space or capitalisation in the model's
output breaks the parser. That fragility is the reason we move to structured tool calls in 2d.

**Run the cell and watch the trace.** Pay attention to:
- The `Thought →  Action → Observation` cycle repeating
- How the model's reasoning changes after each observation
- The `✓ Final Answer` line that ends the loop


In [ ]:
REACT_SYSTEM = """You are a reasoning agent controlling a TurtleSim robot.
You operate in an 11 × 11 metre arena. The turtle starts at x=5.5, y=5.5.

You have these tools:
  get_pose()                    — returns current (x, y, heading_deg)
  move_forward(distance_m)      — moves the turtle forward by distance_m metres
  rotate(angle_deg)             — rotates the turtle; positive = counter-clockwise

Output exactly ONE Thought/Action pair, then stop and wait.
Do not chain multiple actions in a single reply.

Respond ONLY in this exact format, one step at a time:

Thought: <your reasoning>
Action: <tool name>
Action Input: <argument or empty>

When you have a final answer, write:
Thought: I now know the answer.
Final Answer: <your answer>

Begin."""

def fake_get_pose() -> str:
    return "x=5.5, y=5.5, heading_deg=0.0"

def fake_move_forward(distance_m: float) -> str:
    return f"Moved forward {distance_m} m. New pose: x=5.5, y={5.5 + distance_m:.2f}, heading_deg=0.0"

def fake_rotate(angle_deg: float) -> str:
    return f"Rotated {angle_deg}°. New heading: {angle_deg % 360:.1f}°"

TOOL_MAP = {
    "get_pose":     lambda arg: fake_get_pose(),
    "move_forward": lambda arg: fake_move_forward(float(arg)),
    "rotate":       lambda arg: fake_rotate(float(arg)),
}

from dataclasses import dataclass, field

@dataclass
class ReactResult:
    answer: str
    steps_taken: int
    success: bool
    history: list = field(default_factory=list)

def run_react_loop(task: str, max_steps: int = 8, verbose: bool = False) -> ReactResult:
    llm = get_llm()
    history = [SystemMessage(REACT_SYSTEM), HumanMessage(f"Task: {task}")]

    for step in range(max_steps):
        response = llm.invoke(history)
        text = response.text.strip()

        if verbose:
            print(f"\n── Step {step + 1} {'─' * 40}")
            print(text)

        history.append(AIMessage(text))

        for line in text.splitlines():
            if line.startswith("Final Answer:"):
                answer = line.removeprefix("Final Answer:").strip()
                if verbose:
                    print(f"\n✓ Final Answer: {answer}")
                return ReactResult(answer, step + 1, True, history)

        action = action_input = None
        for line in text.splitlines():
            if line.startswith("Action:") and action is None:
                action = line.removeprefix("Action:").strip().rstrip("()")
            if line.startswith("Action Input:") and action is not None:
                action_input = line.removeprefix("Action Input:").strip()
                break

        if not action:
            obs = "[PARSE ERROR] Your response must contain exactly one Thought/Action pair, or a Final Answer. Please retry."
            if verbose:
                print(f"\n  ⚠ {obs}")
            history.append(HumanMessage(f"Observation: {obs}"))
            continue

        if action in TOOL_MAP:\
            try:
                observation = TOOL_MAP[action](action_input or "")
            except (ValueError, TypeError) as e:
                observation = f"[TOOL ERROR] Invalid input for '{action}': {e}"
        else:
            observation = f"[UNKNOWN TOOL] '{action}' does not exist. Available: {list(TOOL_MAP.keys())}"

        if verbose:
            print(f"\n  → Observation: {observation}")
        history.append(HumanMessage(f"Observation: {observation}"))

    return ReactResult("Max steps reached without a final answer.", max_steps, False, history)


print("Running ReAct loop...")
result = run_react_loop(
    "What is the turtle's current pose? Then move it forward 2 metres.",
    verbose=True
)

Running ReAct loop...

── Step 1 ────────────────────────────────────────
Thought: I need to find the current pose of the turtle as requested.
Action: get_pose()
Action Input:

  → Observation: x=5.5, y=5.5, heading_deg=0.0

── Step 2 ────────────────────────────────────────
Thought: The current pose is x=5.5, y=5.5, heading_deg=0.0. Now I need to move the turtle forward 2 metres.
Action: move_forward(distance_m)
Action Input: 2

  → Observation: [UNKNOWN TOOL] 'move_forward(distance_m' does not exist. Available: ['get_pose', 'move_forward', 'rotate']

── Step 3 ────────────────────────────────────────
Thought: I made a mistake in the tool call format. I should use the tool name `move_forward` and provide the value `2` as the input.
Action: move_forward
Action Input: 2

  → Observation: Moved forward 2.0 m. New pose: x=5.5, y=7.50, heading_deg=0.0

── Step 4 ────────────────────────────────────────
Thought: I have retrieved the initial pose (x=5.5, y=5.5, heading_deg=0.0) and moved the

## Exercise 2b — Inspect the growing context window

Every Thought / Action / Observation cycle appends messages to `history`.
That list **is** the agent's memory — there is nothing else.

Run the cell below, then answer:
- How many messages are in history after the loop finishes?
- How does that break down by type?
- How many characters is the full context?

Then re-run `run_react_loop` with `max_steps=1` and compare.


In [3]:
history_debug = result.history   # message list from exercise 2a

print(f"Messages in history : {len(history_debug)}")
print(f"Types               : {[type(m).__name__ for m in history_debug]}")
print(f"Approx chars        : {sum(len(m.text) for m in history_debug)}")
print()
for i, m in enumerate(history_debug):
    preview = m.text[:60].replace('\n', ' ')
    print(f"  [{i}] {type(m).__name__:<15} {preview}...")

Messages in history : 9
Types               : ['SystemMessage', 'HumanMessage', 'AIMessage', 'HumanMessage', 'AIMessage', 'HumanMessage', 'AIMessage', 'HumanMessage', 'AIMessage']
Approx chars        : 1767

  [0] SystemMessage   You are a reasoning agent controlling a TurtleSim robot. You...
  [1] HumanMessage    Task: What is the turtle's current pose? Then move it forwar...
  [2] AIMessage       Thought: I need to find the current pose of the turtle as re...
  [3] HumanMessage    Observation: x=5.5, y=5.5, heading_deg=0.0...
  [4] AIMessage       Thought: The current pose is x=5.5, y=5.5, heading_deg=0.0. ...
  [5] HumanMessage    Observation: [UNKNOWN TOOL] 'move_forward(distance_m' does n...
  [6] AIMessage       Thought: I made a mistake in the tool call format. I should ...
  [7] HumanMessage    Observation: Moved forward 2.0 m. New pose: x=5.5, y=7.50, h...
  [8] AIMessage       Thought: I have retrieved the initial pose (x=5.5, y=5.5, he...


**What to notice:**
- Each round-trip adds one `AIMessage` (the model's Thought/Action) and one `HumanMessage` (the Observation). Two messages per step.
- The `SystemMessage` is the only thing that never changes — it is the agent's fixed context.
- A 5-step task accumulates 12 messages before the final answer. Context windows have limits;
  long-running agents eventually run out of space. This is one reason the planning pattern
  (notebook 04) exists — it uses one LLM call up front instead of one per step.


---
## Breakdown — from ReAct to agent design patterns

The loop you just ran reduces to three components: a model that **thinks**, a set of **actions** it can take, and a **memory** of what has happened.

That simplicity comes with limitations. The agent is reactive — it decides one step at a time, with no broader plan, no way to verify its own output, and no access to other agents. The four patterns below each address one of those limitations.

### 1. Tools
The immediate bottleneck is the quality of available actions. `get_pose()` and `move_forward()` are useful stubs, but real agents need to read files, call APIs, query databases, and run code. **Tools** are well-defined, typed, testable functions the model can invoke. The key design question is not how to call them — it is what makes a good tool: narrow scope, predictable outputs, and errors the agent can recover from gracefully.

### 2. Planning
A ReAct agent commits to each action without knowing what comes next. For short tasks that is acceptable; for longer tasks, or where an early mistake is costly, it is better to have the model produce a plan before it begins acting. Planning separates *what to do* from *how to do it*, producing a structure that can be inspected and corrected before any tool is called.

### 3. Multi-agent workflows
The first three patterns assume a single agent handling everything. Some tasks are too large or varied for that to be practical. Multi-agent systems divide the work: an orchestrator maintains the plan and delegates subtasks to specialist agents, each with their own tools and context window. This decomposition also contains failures — a broken subagent returns an error the orchestrator can handle, rather than corrupting the entire pipeline.

---

**This session covers patterns 1 and 2.** The exercises below replace the fragile text-parsing loop with structured tool-calling, which is the approach used in notebooks 03 and 04.

## Exercise 2c — Define tools with `@tool`

In exercise 2a, the model described its actions as plain text (`Action: move_forward`)
and we parsed that text ourselves. That works, but breaks easily.

The `@tool` decorator gives us something better: it turns a Python function into
a structured object with a name, a description, and a typed argument schema.
The model uses that schema to produce structured JSON output instead of free text.

**The decorator does three things automatically:**
1. `tool.name` — taken from the function name
2. `tool.description` — taken from the **docstring** ← this is what the model reads
3. `tool.args` — a JSON schema built from the **type annotations**

The model never sees your Python code. It only sees name, description, and schema.
Write those three things as carefully as you would write an API contract.

**Run the cell and read the printed output** — that is exactly what the model receives
when you bind these tools in exercise 2d.


In [4]:
from langchain_core.tools import tool

@tool
def get_pose() -> str:
    """Return the turtle's current pose as (x, y, heading_deg). No arguments required."""
    return "x=5.5, y=5.5, heading_deg=0.0"

@tool
def move_forward(distance_m: float) -> str:
    """Move the turtle forward by distance_m metres along its current heading.

    Args:
        distance_m: Distance in metres. Positive only.
    """
    return f"Moved {distance_m} m forward."

@tool
def rotate(angle_deg: float) -> str:
    """Rotate the turtle in place.

    Args:
        angle_deg: Rotation in degrees. Positive = counter-clockwise.
    """
    return f"Rotated {angle_deg}°."


TOOLS = [move_forward, get_pose, rotate]

print("Tool schemas visible to the model:\n")
for t in TOOLS:
    print(f"  {t.name}")
    print(f"    description : {t.description}")
    print(f"    args schema : {t.args}")
    print()

Tool schemas visible to the model:

  move_forward
    description : Move the turtle forward by distance_m metres along its current heading.

Args:
    distance_m: Distance in metres. Positive only.
    args schema : {'distance_m': {'title': 'Distance M', 'type': 'number'}}

  get_pose
    description : Return the turtle's current pose as (x, y, heading_deg). No arguments required.
    args schema : {}

  rotate
    description : Rotate the turtle in place.

Args:
    angle_deg: Rotation in degrees. Positive = counter-clockwise.
    args schema : {'angle_deg': {'title': 'Angle Deg', 'type': 'number'}}



### What to observe
- The description is the *only* documentation the model has.
  Vague descriptions → wrong argument values.
- Unit ambiguity is a common bug: does `distance` mean metres or cells?
  Always name the unit in the argument (`distance_m`, `angle_deg`).
- The JSON schema is what the model fills in when it calls the tool.
  Missing annotations → missing schema → model guesses the type.

## Exercise 2d — Input validation with Pydantic

The `@tool` decorator accepts a Pydantic `BaseModel` as the argument schema.
This lets you add range checks, aliases, and default values — and catch
bad model outputs *before* they reach the robot.

In [5]:
# ── Exercise 2d ──────────────────────────────────────────────────────────────
# Validated tool arguments.
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass, field


class MoveArgs(BaseModel):
    distance_m: float = Field(
        ...,
        description="Distance to move forward in metres.",
        gt=0,
        le=10,
    )


# Test validation directly — no agent involved
try:
    args = MoveArgs(distance_m=-1.0)          # should fail
except Exception as e:
    print(f"Caught bad distance  : {e}")

Caught bad distance  : 1 validation error for MoveArgs
distance_m
  Input should be greater than 0 [type=greater_than, input_value=-1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than


## Exercise 2e — Bind tools to the model

`llm.bind_tools(tools)` does two things:
1. Sends the tool schemas to the model so it knows what is available
2. Switches the model's output format from free text to structured `tool_calls`

Instead of parsing `Action: move_forward` from a string,
you now get back a Python dict: `{'name': 'move_forward', 'args': {'distance_m': 2.0}, 'id': '...'}`.
No parsing required. No fragile string matching.

**Run the cell.** Look at the two printed values:
- `response.text` — often empty when the model wants to call a tool
- `response.tool_calls` — a list of structured call requests

The `id` field on each tool call is important: when you return the result,
you wrap it in a `ToolMessage` with the same `tool_call_id` so the model knows
which call it belongs to.


In [6]:
llm = get_llm()
llm_with_tools = llm.bind_tools(TOOLS)

# Check that we did not need to include the tools in the system prompt for the model to know about them.
print(llm_with_tools.kwargs.get("tools"))

messages = [
    SystemMessage("You are controlling a TurtleSim robot. Use your tools to answer the user's request."),
    HumanMessage("Move turtle 1m forward and tell me its new pose."),
]

response = llm_with_tools.invoke(messages)

print("response.text      :", response.text)
print("response.tool_calls:", response.tool_calls)
print()

# Manually execute the tool call so you can see the round-trip
if response.tool_calls:
    for tc in response.tool_calls:
        print(f"  → Calling : {tc['name']}({tc['args']})")
        match = next((t for t in TOOLS if t.name == tc["name"]), None)
        if match:
            result = match.invoke(tc["args"])
            print(f"  → Result  : {result}")
        print(f"  → ID      : {tc['id']}   ← used to link the ToolMessage")

[{'type': 'function', 'function': {'name': 'move_forward', 'description': 'Move the turtle forward by distance_m metres along its current heading.\n\nArgs:\n    distance_m: Distance in metres. Positive only.', 'parameters': {'properties': {'distance_m': {'type': 'number'}}, 'required': ['distance_m'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'get_pose', 'description': "Return the turtle's current pose as (x, y, heading_deg). No arguments required.", 'parameters': {'properties': {}, 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'rotate', 'description': 'Rotate the turtle in place.\n\nArgs:\n    angle_deg: Rotation in degrees. Positive = counter-clockwise.', 'parameters': {'properties': {'angle_deg': {'type': 'number'}}, 'required': ['angle_deg'], 'type': 'object'}}}]
response.text      : 
response.tool_calls: [{'name': 'move_forward', 'args': {'distance_m': 1}, 'id': 'fe2w5fot', 'type': 'tool_call'}, {'name': 'get_pose', 'args': {}, 'id': 'ju3mmml9

## Exercise 2f — Full tool-calling agent loop *(main activity)*

Now wire the tool-call response into a complete loop. The pattern is:

```
[SystemMessage, HumanMessage]
  → llm_with_tools.invoke()
  → AIMessage  ←  may contain tool_calls
  → for each tool call: run tool → ToolMessage(result, tool_call_id=...)
  → append all ToolMessages to history
  → llm_with_tools.invoke() again
  → repeat until response.tool_calls is empty
```

**The stopping criterion:** when the model returns a response with no `tool_calls`,
it has finished reasoning and is giving its final answer. That empty list is what
ends the loop — there is no special "done" signal, just the absence of more tool calls.
This is also why `max_steps` matters: without it, a confused model that keeps calling
tools would run forever.

**Run `run_tool_agent()` and read the step trace.** Then try changing the task string
and re-running — does the agent adapt its tool call sequence?


In [ ]:
def run_tool_agent(task: str, max_steps: int = 8, verbose: bool = True) -> str:
    """Agent loop using structured tool calls. Returns the model's final plain-text response."""
    llm = get_llm()
    llm_with_tools = llm.bind_tools(TOOLS)

    history = [
        SystemMessage(
            "You are the reasoning core of a TurtleSim robot in an 11 × 11 m arena. "
            "Use your tools step by step to complete the task. "
            "When done, summarise what you did."
        ),
        HumanMessage(task),
    ]

    for step in range(max_steps):
        response = llm_with_tools.invoke(history)
        history.append(response)

        if verbose:
            print(f"\n── Step {step + 1} ────────────────────────────────────────")
            if response.text:
                print(f"  Model : {response.text}")
            for tc in response.tool_calls:
                print(f"  Tool  : {tc['name']}({tc['args']})")

        # Stopping criterion: no tool calls → model is done
        if not response.tool_calls:
            if verbose:
                print(f"\n✓ Final response: {response.text}")
            return response.text

        # Execute each tool call and return the result as a ToolMessage
        for tc in response.tool_calls:
            tool_fn = next((t for t in TOOLS if t.name == tc["name"]), None)
            if tool_fn:
                observation = tool_fn.invoke(tc["args"])
            else:
                observation = f"Error: unknown tool '{tc['name']}'"

            if verbose:
                print(f"  Obs   : {observation}")

            history.append(ToolMessage(str(observation), tool_call_id=tc["id"]))

    return "Max steps reached."


print("Running structured tool-calling agent...\n")
final = run_tool_agent(
    "Get the turtle's current pose, move it forward 2 metres, then rotate 90 degrees. "
    "Report the final pose."
)

Running structured tool-calling agent...


── Step 1 ────────────────────────────────────────
  Tool  : get_pose({})
  Obs   : x=5.5, y=5.5, heading_deg=0.0

── Step 2 ────────────────────────────────────────
  Tool  : move_forward({'distance_m': 2})
  Obs   : Moved 2.0 m forward.

── Step 3 ────────────────────────────────────────
  Tool  : rotate({'angle_deg': 90})
  Obs   : Rotated 90.0°.

── Step 4 ────────────────────────────────────────
  Tool  : get_pose({})
  Obs   : x=5.5, y=5.5, heading_deg=0.0

── Step 5 ────────────────────────────────────────
  Tool  : get_pose({})
  Obs   : x=5.5, y=5.5, heading_deg=0.0

── Step 6 ────────────────────────────────────────
  Model : I have completed the requested actions:
1. **Initial Pose**: The turtle started at `x=5.5, y=5.5, heading_deg=0.0`.
2. **Movement**: Moved the turtle forward by 2 metres.
3. **Rotation**: Rotated the turtle by 90 degrees.
4. **Final Pose**: The final pose reported by the system is `x=5.5, y=5.5, heading_deg=0.0`

## Exercise 2g — Failure modes (challenge)

Agents fail in predictable ways.  Try each of the following and observe
what happens:

1. **Unknown tool** — add a `HumanMessage("Use the fly() tool")` to the
   history before calling the agent.  Does the model hallucinate a result?

2. **Bad argument type** — call `move_forward` with `distance_m="far"`.
   Does the stub crash?  How would a real ROS tool handle it?

3. **Infinite loop** — remove the `max_steps` ceiling and give the model
   a task it can never complete with the stub tools.
   (Observe: in production, always cap steps and token usage.)

4. **No system prompt** — remove the `SystemMessage` and repeat exercise 2e.
   Does the model still use tools correctly?

**Try Scenario 2 and the rest take-home** 

In [8]:
# ── Exercise 2f ──────────────────────────────────────────────────────────────
# Uncomment one scenario at a time.

# Scenario 1 — unknown tool
# run_tool_agent("Use the fly() tool to reach altitude 10.", max_steps=3)

# Scenario 2 — bad argument type
move_forward.invoke({"distance_m": "far"})

# Scenario 3 — observe max_steps guard
# run_tool_agent("Draw a perfect circle.", max_steps=2)

# Scenario 4 — no system prompt
# llm_bare = get_llm().bind_tools(TOOLS)
# llm_bare.invoke([HumanMessage("Move forward 1 metre.")])

ValidationError: 1 validation error for move_forward
distance_m
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='far', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing

Assume I give a valid type `float' for distance, but it is still an incorrect value. Without the args_schema, this will not fail. But we can improve the tool, such that bad arguments are not passed-in. 

In [9]:
@tool
def move_forward(
    distance_m: float,
) -> str:
    """Move the turtle forward by distance_m metres along its current heading.

    Args:
        distance_m: Distance in metres. Positive only.
    """
    return f"Moved {distance_m} m forward."

print(move_forward.invoke({"distance_m": -10}) ) # should not raise a validation error       

# Argument validation with @tool args_schema
@tool(args_schema=MoveArgs)
def move_forward(
    distance_m: float,
) -> str:
    """Move the turtle forward by distance_m metres along its current heading.

    Args:
        distance_m: Distance in metres. Positive only.
    """
    return f"Moved {distance_m} m forward."

# Re-run the bad argument test
print(move_forward.invoke({"distance_m": -10})  )# should raise a validation error       

Moved -10.0 m forward.


ValidationError: 1 validation error for MoveArgs
distance_m
  Input should be greater than 0 [type=greater_than, input_value=-10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than

## Summary

| Concept | How it works |
|---|---|
| LLM as a function | `llm.invoke(messages)` → `AIMessage` |
| Agent loop | Invoke (Think) → execute tool calls (Act) → append `ToolMessage`s (Observe) → repeat |
| Memory | The `history` list — nothing else |
| Tool schema | `@tool` decorator: name from function, description from docstring, args from annotations |
| Structured calls | `llm.bind_tools(tools)` → `response.tool_calls` is a list of dicts |
| Stopping criterion | Loop exits when `response.tool_calls` is empty — the model signals it is done by not calling anything |
| Safety net | `max_steps` caps the loop; without it a stuck agent runs forever |

**Next:** `03_tools.ipynb` — replace the stubs above with real ROS actions and services,
add input validation so the agent recovers from bad arguments instead of crashing,
and connect the finished tool suite back into this loop.
